# BBLAMultiLabelModel

torch==2.0.1
transformers==4.30.0
pandas==2.0.0
numpy==1.24.0
scikit-learn==1.3.0
pyarrow==12.0.0
tqdm==4.65.0
matplotlib==3.7.0
seaborn==0.12.0

## 1. Data loader

In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from typing import Tuple, List, Dict
import logging
from typing import List

logger = logging.getLogger(__name__)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)

In [2]:
class MultiLabelDataset(Dataset):
    """Custom Dataset for Multi-Label Classification"""
    
    def __init__(self, 
                 file_path: str, 
                 tokenizer,
                 max_length: int = 512,
                 tags_list: List[str] = None):
        
        # Load data
        self.df = pd.read_parquet(file_path)
        self.df['text'] = self.df['title'] + ". " + self.df['question']
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.tags_list = tags_list
        self.tag_to_idx = {tag: idx for idx, tag in enumerate(self.tags_list)}
        
        logger.info(f"Loaded {len(self.df)} samples from {file_path}")
        
    def __len__(self) -> int:
        return len(self.df)
    
    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        """
        Get single sample
        
        Returns:
            Dictionary with:
                - input_ids
                - attention_mask
                - labels (multi-hot encoded)
        """
        
        row = self.df.iloc[idx]
        question = row['text']
        tags = row['tags']
        
        # Convert tags to list if it's numpy array
        if isinstance(tags, np.ndarray):
            tags = tags.tolist()
        
        # Tokenize question
        encoding = self.tokenizer(
            question,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Create multi-hot encoded labels
        labels = np.zeros(len(self.tags_list), dtype=np.float32)
        for tag in tags:
            if tag in self.tag_to_idx:
                labels[self.tag_to_idx[tag]] = 1.0
        
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(labels, dtype=torch.float32)
        }

In [3]:
def create_data_loaders(
    model_path: str,
    train_path: str,
    val_path: str,
    test_path: str,
    tags_list: List,
    max_length: int = 512,
    train_batch_size: int = 32,
    val_batch_size: int = 64,
    test_batch_size: int = 64
) -> Tuple[DataLoader, DataLoader, DataLoader]:
    """
    Create train, validation, and test data loaders
    
    Returns:
        (train_loader, val_loader, test_loader)
    """
    
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    
    # Create datasets
    train_dataset = MultiLabelDataset(
        train_path,
        tokenizer,
        max_length=max_length,
        tags_list=tags_list
    )
    
    val_dataset = MultiLabelDataset(
        val_path,
        tokenizer,
        max_length=max_length,
        tags_list=tags_list
    )
    
    test_dataset = MultiLabelDataset(
        test_path,
        tokenizer,
        max_length=max_length,
        tags_list=tags_list
    )
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=train_batch_size,
        shuffle=True,
        num_workers=0
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=val_batch_size,
        shuffle=False,
        num_workers=0
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=test_batch_size,
        shuffle=False,
        num_workers=0
    )
    
    return train_loader, val_loader, test_loader

## 2. Config

In [4]:
import os

In [5]:
def build_label_mappings(parquet_path: str, label_name: str):
    df = pd.read_parquet(parquet_path)
    if label_name not in df.columns:
        raise ValueError(f"Column {label_name} not found in dataset")
    all_labels = set()
    for array in df[label_name]:
        if isinstance(array, (list, np.ndarray)):
            all_labels.update(array)
    TAGS = sorted(list(all_labels))
    TAG_TO_IDX = {tag: idx for idx, tag in enumerate(TAGS)}
    IDX_TO_TAG = {idx: tag for idx, tag in enumerate(TAGS)}

    logger.info(f"TAGS: {TAGS}")
    logger.info(f"TAG_TO_IDX: {TAG_TO_IDX}")
    logger.info(f"IDX_TO_TAG: {IDX_TO_TAG}")

    return TAGS, TAG_TO_IDX, IDX_TO_TAG

In [6]:
def save_label_mappings_txt(
    TAGS,
    TAG_TO_IDX,
    IDX_TO_TAG,
    model_save_path: str
):
    save_dir = os.path.dirname(model_save_path)
    os.makedirs(save_dir, exist_ok=True)
    tags_file = os.path.join(save_dir, "TAGS.txt")
    with open(tags_file, "w", encoding="utf-8") as f:
        for tag in TAGS:
            f.write(f"{tag}\n")
            
    tag_to_idx_file = os.path.join(save_dir, "TAG_TO_IDX.txt")
    with open(tag_to_idx_file, "w", encoding="utf-8") as f:
        for tag, idx in TAG_TO_IDX.items():
            f.write(f"{tag}\t{idx}\n")
            
    idx_to_tag_file = os.path.join(save_dir, "IDX_TO_TAG.txt")
    with open(idx_to_tag_file, "w", encoding="utf-8") as f:
        for idx, tag in IDX_TO_TAG.items():
            f.write(f"{idx}\t{tag}\n")

    logger.info("Saved label mappings:")
    logger.info(f"  TAGS -> {tags_file}")
    logger.info(f"  TAG_TO_IDX -> {tag_to_idx_file}")
    logger.info(f"  IDX_TO_TAG -> {idx_to_tag_file}")

In [7]:
class Config:
    """Configuration for Multi-Label Classification Model"""
    
    # Data
    TRAIN_PATH = "/kaggle/input/datasets/kietnguyenoto/data-1024-100k-10-tags/train.parquet"
    VAL_PATH = "/kaggle/input/datasets/kietnguyenoto/data-1024-100k-10-tags/val.parquet"
    TEST_PATH = "/kaggle/input/datasets/kietnguyenoto/data-1024-100k-10-tags/test.parquet"
    
    TAGS, TAG_TO_IDX, IDX_TO_TAG = build_label_mappings(parquet_path=TRAIN_PATH, label_name='tags')
    NUM_TAGS = len(TAGS)
    
    # Model
    MODEL_PATH = "/kaggle/input/models/jsday96/codebert-base/transformers/default/1/codebert-base"
    CODEBERT_HIDDEN_SIZE = 768
    LSTM_HIDDEN_SIZE = 512
    NUM_ATTENTION_HEADS = 8
    DROPOUT = 0.2
    
    # Training
    TRAIN_BATCH_SIZE = 32
    VAL_BATCH_SIZE = 64
    TEST_BATCH_SIZE = 64
    NUM_EPOCHS = 10
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    
    # Device
    DEVICE = "cuda"  # or "cpu"
    
    # Prediction threshold
    PREDICTION_THRESHOLD = 0.5
    
    # Others
    MAX_LENGTH = 512
    SEED = 42
    SAVE_PATH = "./models/bbla_model.pt"
    save_label_mappings_txt(TAGS, TAG_TO_IDX, IDX_TO_TAG, SAVE_PATH)

2026-06-12 20:37:11,951 - __main__ - INFO - TAGS: ['android', 'c#', 'c++', 'html', 'ios', 'java', 'javascript', 'jquery', 'php', 'python']
2026-06-12 20:37:11,951 - __main__ - INFO - TAG_TO_IDX: {'android': 0, 'c#': 1, 'c++': 2, 'html': 3, 'ios': 4, 'java': 5, 'javascript': 6, 'jquery': 7, 'php': 8, 'python': 9}
2026-06-12 20:37:11,951 - __main__ - INFO - IDX_TO_TAG: {0: 'android', 1: 'c#', 2: 'c++', 3: 'html', 4: 'ios', 5: 'java', 6: 'javascript', 7: 'jquery', 8: 'php', 9: 'python'}
2026-06-12 20:37:11,968 - __main__ - INFO - Saved label mappings:
2026-06-12 20:37:11,968 - __main__ - INFO -   TAGS -> ./models/TAGS.txt
2026-06-12 20:37:11,968 - __main__ - INFO -   TAG_TO_IDX -> ./models/TAG_TO_IDX.txt
2026-06-12 20:37:11,968 - __main__ - INFO -   IDX_TO_TAG -> ./models/IDX_TO_TAG.txt


## 3. Model

In [8]:
import torch
import torch.nn as nn
from transformers import AutoModel

In [9]:
import torch
import torch.nn as nn
import math
import torch.nn.functional as F


class RMSNorm(nn.Module):
    """Gated RMS Normalization - tương tự gated_rms_norm trong code TensorFlow"""
    
    def __init__(self, dim: int, eps: float = 1e-8):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
        self.gate = nn.Parameter(torch.ones(dim))
    
    def forward(self, x):
        # RMS normalization
        rms = torch.sqrt(torch.mean(x ** 2, dim=-1, keepdim=True) + self.eps)
        normalized = x / rms
        
        # Gated mechanism
        return normalized * self.weight * torch.sigmoid(self.gate)


class CustomMultiHeadAttention(nn.Module):
    """
    Multi-Head Attention với ReLU thay vì Softmax
    Dựa trên cấu trúc dot_attention từ TensorFlow
    
    :param embed_dim: embedding dimension
    :param num_heads: số attention heads
    :param hidden_size: attention space dimension (mặc định = embed_dim)
    :param dropout: attention dropout rate
    :param use_layer_norm: có sử dụng layer normalization hay không
    :param use_out_projection: có sử dụng output projection hay không
    :param use_rms_norm: có sử dụng RMSNorm post-normalization hay không
    """

    def __init__(
        self,
        embed_dim: int,
        num_heads: int,
        hidden_size: int = None,
        dropout: float = 0.0,
        use_layer_norm: bool = False,
        use_out_projection: bool = True,
        use_rms_norm: bool = True
    ):
        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.hidden_size = hidden_size or embed_dim
        self.head_dim = self.hidden_size // num_heads
        self.use_out_projection = use_out_projection
        self.use_rms_norm = use_rms_norm
        self.dropout_rate = dropout

        assert self.head_dim * num_heads == self.hidden_size, \
            f"hidden_size ({self.hidden_size}) phải chia hết cho num_heads ({num_heads})"

        # Query, Key, Value projections
        self.q_proj = nn.Linear(embed_dim, self.hidden_size)
        self.k_proj = nn.Linear(embed_dim, self.hidden_size)
        self.v_proj = nn.Linear(embed_dim, self.hidden_size)

        # Optional layer normalization
        self.use_layer_norm = use_layer_norm
        if use_layer_norm:
            self.ln_q = nn.LayerNorm(self.hidden_size)
            self.ln_k = nn.LayerNorm(self.hidden_size)
            self.ln_v = nn.LayerNorm(self.hidden_size)

        # Post-attention RMSNorm (thay vì layer norm thông thường)
        if use_rms_norm:
            self.rms_norm = RMSNorm(embed_dim)
        
        # Output projection (tương tự o_map trong TensorFlow)
        if use_out_projection:
            self.out_proj = nn.Linear(embed_dim, embed_dim)

        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """
        Chia x thành multiple heads
        Input: [batch_size, seq_len, hidden_size]
        Output: [batch_size, num_heads, seq_len, head_dim]
        """
        batch_size, seq_len, _ = x.shape
        x = x.view(batch_size, seq_len, self.num_heads, self.head_dim)
        return x.transpose(1, 2)

    def combine_heads(self, x: torch.Tensor) -> torch.Tensor:
        """
        Kết hợp multiple heads lại
        Input: [batch_size, num_heads, seq_len, head_dim]
        Output: [batch_size, seq_len, hidden_size]
        """
        batch_size, num_heads, seq_len, head_dim = x.shape
        x = x.transpose(1, 2).contiguous()
        return x.view(batch_size, seq_len, num_heads * head_dim)

    def forward(
        self,
        query,
        key=None,
        value=None,
        key_padding_mask=None,
        cache=None,
        return_cache=False
    ):
        """
        Forward pass cho attention mechanism
        
        :param query: [batch_size, query_len, embed_dim]
        :param key: [batch_size, seq_len, embed_dim] hoặc None (self-attention)
        :param value: [batch_size, seq_len, embed_dim] hoặc None (self-attention)
        :param key_padding_mask: [batch_size, seq_len] - 0 cho padding, 1 cho valid tokens
        :param cache: Cache từ lần forward trước (cho decoding)
        :param return_cache: Có trả lại cache hay không
        :return: output và attention weights (hoặc kèm cache)
        """
        batch_size = query.size(0)

        # Self-attention: query = key = value
        if key is None:
            key = query
        if value is None:
            value = query

        # ========== Tính Q, K, V ==========
        q = self.q_proj(query)
        k = self.k_proj(key)
        v = self.v_proj(value)

        # Optional layer normalization
        if self.use_layer_norm:
            q = self.ln_q(q)
            k = self.ln_k(k)
            v = self.ln_v(v)

        # Xử lý cache (tương tự cache handling trong TensorFlow code)
        if cache is not None:
            k = torch.cat([cache.get('k', torch.tensor([])), k], dim=1)
            v = torch.cat([cache.get('v', torch.tensor([])), v], dim=1)

        # Chia thành multiple heads
        q = self.split_heads(q)  # [batch_size, num_heads, query_len, head_dim]
        k = self.split_heads(k)  # [batch_size, num_heads, seq_len, head_dim]
        v = self.split_heads(v)  # [batch_size, num_heads, seq_len, head_dim]

        # ========== Scaled dot-product attention ==========
        # Scale query (tương tự q *= (hidden_size // num_heads) ** (-0.5))
        q = q * (self.head_dim ** (-0.5))

        # Q * K^T => attention logits
        logits = torch.matmul(q, k.transpose(-2, -1))

        # ========== Áp dụng mask ==========
        if key_padding_mask is not None:
            # Chuyển mask từ [batch_size, seq_len] thành [batch_size, 1, 1, seq_len]
            # Đảo ngược: 0 -> 1 (mask), 1 -> 0 (không mask)
            # mask = (1.0 - key_padding_mask.unsqueeze(1).unsqueeze(2)).bool()
            mask = key_padding_mask.unsqueeze(1).unsqueeze(2)
            logits = logits.masked_fill(mask, float('-inf'))

        # ========== Thay softmax bằng ReLU (như TensorFlow code) ==========
        weights = F.relu(logits)

        # Dropout
        weights = self.dropout(weights)

        # ========== Weights * V => attention output ==========
        output = torch.matmul(weights, v)

        # Kết hợp multiple heads
        output = self.combine_heads(output)  # [batch_size, query_len, hidden_size]

        # ========== Post-attention normalization ==========
        # RMSNorm để ổn định (tương tự gated_rms_norm trong TensorFlow)
        if self.use_rms_norm:
            output = self.rms_norm(output)

        # ========== Output projection ==========
        if self.use_out_projection:
            output = self.out_proj(output)

        # # Chuẩn bị results
        # results = {
        #     'output': output,
        #     'weights': weights,
        # }

        # # Trả lại cache nếu được yêu cầu
        # if return_cache:
        #     results['cache'] = {'k': k, 'v': v}

        return output, weights

In [10]:
class BBLAMultiLabelModel(nn.Module):
    """
    BERT-Bi-LSTM-Attention Multi-Label Classification Model
    
    Architecture:
    CodeBERT → Bi-LSTM → Multi-Head Attention → 10 Classification Heads
    """
    
    def __init__(self, 
                 model_path: str = "microsoft/codebert-base",
                 lstm_hidden: int = 512,
                 num_tags: int = 10,
                 num_attention_heads: int = 4,
                 dropout: float = 0.2):
        super().__init__()
        
        # 1. CodeBERT (Pretrained)
        self.codebert_model = AutoModel.from_pretrained(model_path)
        codebert_hidden = self.codebert_model.config.hidden_size  # 768
        
        # Freeze CodeBERT layers to save memory and training time
        for param in self.codebert_model.parameters():
            param.requires_grad = False
        
        # 2. Bi-LSTM (2 layers)
        self.bilstm = nn.LSTM(
            input_size=codebert_hidden,           # 768
            hidden_size=lstm_hidden,              # 512
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        
        bilstm_output_size = lstm_hidden * 2     # 1024 (512 * 2 directions)
        
        # 3. Multi-Head Self-Attention
        self.attention = CustomMultiHeadAttention(
                    embed_dim=bilstm_output_size,         # 1024
                    num_heads=num_attention_heads,        # 4
                    dropout=dropout
                )
        
        # 4. Layer Normalization and Feed-Forward (Optional but recommended)
        self.norm = nn.LayerNorm(bilstm_output_size)
        self.feed_forward = nn.Sequential(
            nn.Linear(bilstm_output_size, bilstm_output_size * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(bilstm_output_size * 2, bilstm_output_size)
        )
        self.norm2 = nn.LayerNorm(bilstm_output_size)
        
        # 5. Classification Heads (10 independent heads for each tag)
        self.classification_heads = nn.ModuleList([
            nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(bilstm_output_size, 256),
                nn.ReLU(),
                nn.BatchNorm1d(256),
                nn.Dropout(dropout),
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(128, 1),
                nn.Sigmoid()
            )
            for _ in range(num_tags)
        ])
        
        self.num_tags = num_tags
        
    def forward(self, 
                input_ids: torch.Tensor, 
                attention_mask: torch.Tensor) -> torch.Tensor:
        """
        Forward pass
        
        Args:
            input_ids: [batch_size, seq_len]
            attention_mask: [batch_size, seq_len]
            
        Returns:
            predictions: [batch_size, num_tags] - probabilities (0-1)
        """
        
        # Step 1: CodeBERT embeddings
        with torch.no_grad():
            codebert_output = self.codebert_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True
            )
            embeddings = codebert_output.last_hidden_state  # [batch, seq_len, 768]
        
        # Step 2: Bi-LSTM
        lstm_out, (h_n, c_n) = self.bilstm(embeddings)
        # lstm_out: [batch_size, seq_len, 1024]
        
        # Step 3: Multi-Head Self-Attention
        # Create padding mask for attention
        src_key_padding_mask = ~attention_mask.bool()  # Invert mask
        
        attention_out, attention_weights = self.attention(
            lstm_out, 
            lstm_out, 
            lstm_out,
            key_padding_mask=src_key_padding_mask
        )
        # attention_out: [batch_size, seq_len, 1024]
        
        # Add & Norm (Residual connection)
        lstm_out = self.norm(lstm_out + attention_out)
        
        # Feed-Forward Network
        ff_out = self.feed_forward(lstm_out)
        lstm_out = self.norm2(lstm_out + ff_out)
        # lstm_out: [batch_size, seq_len, 1024]
        
        # Step 4: Pooling - Take last token
        pooled_output = lstm_out[:, -1, :]  # [batch_size, 1024]
        
        # Step 5: Classification Heads (10 heads for 10 tags)
        outputs = []
        for head in self.classification_heads:
            output = head(pooled_output)  # [batch_size, 1]
            outputs.append(output)
        
        predictions = torch.cat(outputs, dim=1)  # [batch_size, num_tags]
        
        return predictions

## 4. Trainer

In [11]:
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
import numpy as np
from tqdm import tqdm
from typing import Dict
import os
import time
import logging

logger = logging.getLogger(__name__)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)

In [12]:
class Trainer:
    """Trainer class for multi-label classification model"""
    
    def __init__(self, 
                 model: nn.Module,
                 device: str = 'cuda',
                 save_path: str = './models/best_model.pt'):
        self.model = model.to(device)
        self.device = device
        self.save_path = save_path
        
        # Create directory if not exists
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        
        # Loss function for multi-label: BCELoss (Binary Cross Entropy)
        self.criterion = nn.BCELoss()
        
        self.best_f1 = 0
        self.patience_counter = 0
        
    def train_epoch(self,
                   train_loader: DataLoader,
                   optimizer: torch.optim.Optimizer,
                ) -> float:
        """Train for one epoch"""
        
        self.model.train()
        total_loss = 0
        
        progress_bar = tqdm(train_loader, desc="Training")
        
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(self.device)
            attention_mask = batch['attention_mask'].to(self.device)
            labels = batch['labels'].to(self.device)
            
            # Forward pass
            predictions = self.model(input_ids, attention_mask)
            
            # Calculate loss
            loss = self.criterion(predictions, labels)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item()
            progress_bar.set_postfix({'loss': loss.item()})
        
        avg_loss = total_loss / len(train_loader)
        return avg_loss
    
    def evaluate(self,
                val_loader: DataLoader,
                threshold: float = 0.5,
        ) -> Dict[str, float]:
        """
        Evaluate model on validation/test set
        
        Returns:
            Dictionary with metrics: accuracy, precision, recall, f1
        """
        
        self.model.eval()
        all_predictions = []
        all_labels = []
        total_loss = 0
        start_time = time.time()
        total_samples = 0
            
        with torch.no_grad():
            progress_bar = tqdm(val_loader, desc="Evaluating")
            
            for batch in progress_bar:
                input_ids = batch['input_ids'].to(self.device)
                attention_mask = batch['attention_mask'].to(self.device)
                labels = batch['labels'].to(self.device)

                batch_size = labels.size(0)
                total_samples += batch_size
                
                # Forward pass
                predictions = self.model(input_ids, attention_mask)
                
                # Loss
                loss = self.criterion(predictions, labels)
                total_loss += loss.item()
                
                # Convert to binary predictions
                binary_preds = (predictions > threshold).cpu().numpy()
                
                all_predictions.extend(binary_preds)
                all_labels.extend(labels.cpu().numpy())
                
                progress_bar.set_postfix({'loss': loss.item()})

        total_time = time.time() - start_time
        all_predictions = np.array(all_predictions)
        all_labels = np.array(all_labels)
        
        # Calculate metrics
        metrics = self.calculate_metrics(all_predictions, all_labels)
        metrics['loss'] = total_loss / len(val_loader)

        metrics['total_inference_time_sec'] = total_time
        metrics['avg_inference_time_per_sample_ms'] = (
            total_time / total_samples
        ) * 1000
        
        return metrics
    
    @staticmethod
    def calculate_metrics(predictions: np.ndarray, 
                         labels: np.ndarray) -> Dict[str, float]:
        """
        Calculate metrics for multi-label classification
        
        Args:
            predictions: [num_samples, num_tags] binary predictions
            labels: [num_samples, num_tags] ground truth labels
        """
        
        # Exact Match Ratio (Hamming Loss)
        exact_match = np.mean(np.all(predictions == labels, axis=1))
        
        # Subset Accuracy
        subset_accuracy = np.mean(
            (predictions == labels).sum(axis=1) == predictions.shape[1]
        )
        
        # Per-label metrics
        tp = np.sum(predictions * labels, axis=0)
        fp = np.sum(predictions * (1 - labels), axis=0)
        fn = np.sum((1 - predictions) * labels, axis=0)
        
        # Precision, Recall, F1 per label
        precision = tp / (tp + fp + 1e-8)
        recall = tp / (tp + fn + 1e-8)
        f1 = 2 * (precision * recall) / (precision + recall + 1e-8)
        
        # Macro-averaged metrics
        macro_precision = np.mean(precision)
        macro_recall = np.mean(recall)
        macro_f1 = np.mean(f1)
        
        # Micro-averaged metrics
        micro_tp = np.sum(tp)
        micro_fp = np.sum(fp)
        micro_fn = np.sum(fn)
        
        micro_precision = micro_tp / (micro_tp + micro_fp + 1e-8)
        micro_recall = micro_tp / (micro_tp + micro_fn + 1e-8)
        micro_f1 = 2 * (micro_precision * micro_recall) / (
            micro_precision + micro_recall + 1e-8
        )
        
        return {
            'exact_match': exact_match,
            'subset_accuracy': subset_accuracy,
            'macro_precision': macro_precision,
            'macro_recall': macro_recall,
            'macro_f1': macro_f1,
            'micro_precision': micro_precision,
            'micro_recall': micro_recall,
            'micro_f1': micro_f1,
        }
    
    def save_model(self):
        """Save model checkpoint"""
        torch.save(self.model.state_dict(), self.save_path)
        print(f"Model saved to {self.save_path}")
    
    def load_model(self):
        """Load model checkpoint"""
        self.model.load_state_dict(torch.load(self.save_path))
        print(f"Model loaded from {self.save_path}")

In [13]:
def train(model: nn.Module,
          train_loader: DataLoader,
          val_loader: DataLoader,
          num_epochs: int = 10,
          device: str = "cuda",
          save_path: str = "./models/best_model.pt",
          learning_rate: float = 1e-4,
          weight_decay: float = 1e-5,
          prediction_threshold: float=0.5
    ) -> Trainer:
    
    trainer = Trainer(model, device=device, save_path=save_path)
    
    optimizer = AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )
    
    logger.info("\n" + "="*80)
    logger.info("Starting Training")
    logger.info("="*80)
    
    for epoch in range(1, num_epochs + 1):
        logger.info(f"\n{'='*80}")
        logger.info(f"Epoch {epoch}/{num_epochs}")
        logger.info(f"{'='*80}")
        
        # Train
        train_loss = trainer.train_epoch(train_loader, optimizer)
        logger.info(f"Train Loss: {train_loss:.4f}")
        
        # Validate
        val_metrics = trainer.evaluate(val_loader, 
                                       threshold=prediction_threshold,
                                       )
        
        logger.info(f"Val Loss: {val_metrics['loss']:.4f}")
        logger.info(f"Val Macro F1: {val_metrics['macro_f1']:.4f}")
        logger.info(f"Val Micro F1: {val_metrics['micro_f1']:.4f}")
        logger.info(f"Val Exact Match: {val_metrics['exact_match']:.4f}")
        
        # Save best model
        if val_metrics['macro_f1'] > trainer.best_f1:
            trainer.best_f1 = val_metrics['macro_f1']
            trainer.save_model()
            trainer.patience_counter = 0
            logger.info("Best model saved!")
        else:
            trainer.patience_counter += 1
            logger.info(f"Patience: {trainer.patience_counter}")
    
    logger.info("\n" + "="*80)
    logger.info("Training completed!")
    logger.info("="*80)
    
    return trainer

## 5. Main

In [14]:
import torch
import numpy as np
import os
# warnings.filterwarnings('ignore')

In [15]:
def set_seed(seed: int = 42):
    """Set random seed for reproducibility"""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [16]:
def save_test_metrics_to_csv(test_metrics: dict, file_path: str):
    df = pd.DataFrame([test_metrics])

    if not os.path.exists(file_path):
        df.to_csv(file_path, index=False)
    else:
        df.to_csv(file_path, mode='a', header=False, index=False)

    print(f"Saved test metrics to: {file_path}")

In [17]:
def main():
    # Set seed
    config_obj = Config()
    set_seed(config_obj.SEED)
    
    # Check device
    print(f"Using device: {config_obj.DEVICE}")
    if config_obj.DEVICE == 'cuda':
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    
    print("\n" + "="*80)
    print("Multi-Label Classification with Attention-based Multi-Head")
    print("="*80)
    
    # Create data loaders
    print("\nLoading data...")
    train_loader, val_loader, test_loader = create_data_loaders(
        model_path=config_obj.MODEL_PATH,
        train_path=config_obj.TRAIN_PATH,
        val_path=config_obj.VAL_PATH,
        test_path=config_obj.TEST_PATH,
        tags_list=config_obj.TAGS,
        max_length=config_obj.MAX_LENGTH,
        train_batch_size=config_obj.TRAIN_BATCH_SIZE,
        val_batch_size=config_obj.VAL_BATCH_SIZE,
        test_batch_size=config_obj.TEST_BATCH_SIZE
    )
    
    # Create model
    print("\nCreating model...")
    model = BBLAMultiLabelModel(
        model_path=config_obj.MODEL_PATH,
        lstm_hidden=config_obj.LSTM_HIDDEN_SIZE,
        num_tags=config_obj.NUM_TAGS,
        num_attention_heads=config_obj.NUM_ATTENTION_HEADS,
        dropout=config_obj.DROPOUT
    )
    
    # Print model summary
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    
    # Train
    print("\nTraining model...")
    start_time = time.time()
    trainer = train(
        model=model, 
        train_loader=train_loader, 
        val_loader=val_loader,
        num_epochs=config_obj.NUM_EPOCHS,
        device=config_obj.DEVICE,
        save_path=config_obj.SAVE_PATH,
        learning_rate=config_obj.LEARNING_RATE,
        weight_decay=config_obj.WEIGHT_DECAY,
        prediction_threshold=config_obj.PREDICTION_THRESHOLD
    )
    train_time = time.time() - start_time
    
    # Load best model
    print("\nLoading best model...")
    trainer.load_model()
    
    # Evaluate on test set
    print("\nEvaluating on test set...")
    test_metrics = trainer.evaluate(test_loader, 
                                    threshold=config_obj.PREDICTION_THRESHOLD,
                                )
    
    print("\n" + "="*80)
    print("TEST SET RESULTS")
    print("="*80)
    print(f"Test Loss:           {test_metrics['loss']:.4f}")
    print(f"Test Macro Precision: {test_metrics['macro_precision']:.4f}")
    print(f"Test Macro Recall:    {test_metrics['macro_recall']:.4f}")
    print(f"Test Macro F1:        {test_metrics['macro_f1']:.4f}")
    print(f"Test Micro Precision: {test_metrics['micro_precision']:.4f}")
    print(f"Test Micro Recall:    {test_metrics['micro_recall']:.4f}")
    print(f"Test Micro F1:        {test_metrics['micro_f1']:.4f}")
    print(f"Test Exact Match:     {test_metrics['exact_match']:.4f}")
    print(f"total_inference_time_sec: {test_metrics['total_inference_time_sec']: 4f} s")
    print(f"avg inference time per sample ms: {test_metrics['avg_inference_time_per_sample_ms']: 4f} ms")
    print(f"Train time: {train_time: 4f} s")
    print("="*80)
    test_metrics['train_time'] = train_time

    save_test_metrics_to_csv(
        test_metrics,
        file_path="test_results.csv"
    )

In [18]:
main()

Using device: cuda
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition

Multi-Label Classification with Attention-based Multi-Head

Loading data...


2026-06-12 20:37:12,598 - __main__ - INFO - Loaded 80000 samples from /kaggle/input/datasets/kietnguyenoto/data-1024-100k-10-tags/train.parquet
2026-06-12 20:37:12,704 - __main__ - INFO - Loaded 10000 samples from /kaggle/input/datasets/kietnguyenoto/data-1024-100k-10-tags/val.parquet
2026-06-12 20:37:12,788 - __main__ - INFO - Loaded 10000 samples from /kaggle/input/datasets/kietnguyenoto/data-1024-100k-10-tags/test.parquet



Creating model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Total parameters: 147,557,642
Trainable parameters: 22,912,010

Training model...


...

Model saved to ./models/bbla_model.pt


Training: 100%|██████████| 2500/2500 [06:22<00:00,  6.53it/s, loss=0.0238]
2026-06-12 21:45:55,908 - __main__ - INFO - Train Loss: 0.0448
Evaluating: 100%|██████████| 157/157 [00:31<00:00,  5.01it/s, loss=0.0254]
2026-06-12 21:46:27,249 - __main__ - INFO - Val Loss: 0.0481
2026-06-12 21:46:27,249 - __main__ - INFO - Val Macro F1: 0.9056
2026-06-12 21:46:27,249 - __main__ - INFO - Val Micro F1: 0.9170
2026-06-12 21:46:27,249 - __main__ - INFO - Val Exact Match: 0.8649
2026-06-12 21:46:27,249 - __main__ - INFO - Patience: 1
2026-06-12 21:46:27,249 - __main__ - INFO - 
2026-06-12 21:46:27,249 - __main__ - INFO - Training completed!
2026-06-12 21:46:27,249 - __main__ - INFO - ================================================================================



Loading best model...
Model loaded from ./models/bbla_model.pt

Evaluating on test set...


Evaluating: 100%|██████████| 157/157 [00:31<00:00,  5.01it/s, loss=0.0193]


TEST SET RESULTS
Test Loss:           0.0473
Test Macro Precision: 0.9313
Test Macro Recall:    0.8843
Test Macro F1:        0.9065
Test Micro Precision: 0.9323
Test Micro Recall:    0.9024
Test Micro F1:        0.9171
Test Exact Match:     0.8702
total_inference_time_sec:  31.317425 s
avg inference time per sample ms:  3.131743 ms
Train time:  4150.103628 s
Saved test metrics to: test_results.csv
